# Modern Inference Systems

> The inference acceleration section of Part 3 covered operator-level optimizations for a single inference pass — KV Cache, FlashAttention, operator fusion. That was the "single request" perspective.
>
> This section shifts to the perspective of "serving many requests at once." When 100 users simultaneously send requests to the same model, how does a single GPU hold up? The answer is a set of system-level techniques: PagedAttention solves memory fragmentation, continuous batching solves throughput, prefix caching solves repeated prefixes, and prefill/decode disaggregation resolves the resource conflict between the two compute modes. We start from service metrics and unpack each one.

LLM inference serving is different from traditional web serving. Traditional web services have small per-request compute and low latency, so a simple process pool can handle them. LLM inference generates hundreds to thousands of tokens per request, and every token requires a full forward pass, so a single GPU can only serve a limited number of concurrent requests — too many blows up memory, too few kills throughput.

The metrics for service quality are therefore different too. **Throughput** (tokens generated per second) measures total output capacity; **time to first token** (TTFT) measures how long a user waits for the first response; **time per output token** (TPOT) measures smoothness during generation. These three metrics constrain each other and cannot all be optimal at the same time.

The main job of a modern inference system is to find the right balance among these metrics. The following sections unfold in a "problem → solution" order: first the fragmentation problem of KV cache under multiple requests, and how PagedAttention solves it; then how continuous batching pushes throughput to the limit; and finally prefix caching and prefill/decode disaggregation, two more recent optimizations.

## 1. Service Metrics: Throughput, Latency, Concurrency

Let us define the metrics clearly before discussing optimization. LLM serving commonly uses four metrics:

| Metric | Full name | Meaning | Who cares |
|:---|:---|:---|:---|
| **Throughput** | Throughput | Total tokens generated per second (across all requests) | Service operator (cost) |
| **TTFT** | Time To First Token | Time from user request to receiving the first token | User experience |
| **TPOT** | Time Per Output Token | Average interval between tokens during generation | User experience |
| **Concurrency** | Concurrency | Number of requests being served simultaneously | Capacity planning |

These are not independent of each other. Given fixed GPU compute, doubling concurrency roughly doubles throughput, but TPOT also degrades — each request gets less compute. TTFT is dominated by the prefill phase, while TPOT is dominated by the decode phase. Prefill processes the entire prompt in one shot (compute-intensive), while decode generates tokens one by one (bandwidth-intensive). These two compute modes use resources in completely different ways, which Section 6 elaborates on.

Let us use a set of concrete numbers to get a feel for the magnitude of these metrics.

In [ ]:
# Concrete numbers for service metrics
# Assumption: A100 GPU (80GB), Llama-7B FP16, batch=32

gpu_tflops = 312       # A100 FP16 peak compute
params = 7e9           # Llama-7B parameter count
flops_per_token = 2 * 2 * params   # FLOPs for one token forward (2x params x batch, fwd+bwd merged into one inference)
batch_size = 32

# Decode phase: how many forwards per second
# Note: real decode is memory-bound; here we only compute the compute ceiling
flops_per_forward = flops_per_token * batch_size
compute_limit = gpu_tflops * 1e12 / flops_per_forward

print(f"=== A100 + Llama-7B + batch={batch_size} ===")
print(f"GPU compute: {gpu_tflops} TFLOPS (FP16)")
print(f"FLOPs per token forward: {flops_per_token:.2e}")
print(f"FLOPs per forward at batch={batch_size}: {flops_per_forward:.2e}")
print(f"Compute ceiling: {compute_limit:.1f} forwards/s")
print(f"Theoretical throughput ceiling: {compute_limit * batch_size:.0f} tokens/s")
print()

# Real decode is memory-bound, limited by memory bandwidth
gpu_bw_gbs = 2000     # A100 memory bandwidth ~2 TB/s
bytes_per_token = params * 2  # FP16, read weights once
bw_limit = gpu_bw_gbs * 1e9 / bytes_per_token  # how many tokens can be read per second

print(f"Memory bandwidth: {gpu_bw_gbs} GB/s")
print(f"Bytes read per token: {bytes_per_token / 1e9:.1f} GB (weights)")
print(f"Memory ceiling: {bw_limit:.0f} tokens/s (single stream)")
print(f"At batch={batch_size}: {bw_limit * batch_size:.0f} tokens/s (bandwidth reuse)")
print()
print(f"Key observation: the compute ceiling ({compute_limit * batch_size:.0f}) is far above the memory limit")
print(f"The real bottleneck is memory bandwidth — this is why KV cache size matters so much")

## 2. The KV Cache Fragmentation Problem

Recall the KV cache from the Part 3 inference acceleration section: during generation, the K and V of every historical token must be cached for each request. When many requests are served simultaneously, each request's KV cache has a different size — some prompts are short (100 tokens), some are long (10K tokens), and generation lengths vary too.

The most naive memory management is **contiguous allocation**: each request gets a contiguous chunk of GPU memory. This causes two problems. The first is **internal fragmentation**: a request needs 100 tokens, but allocation granularity is per page (say 16 tokens), so it actually gets space for 112 tokens, wasting 12 tokens. The second is **external fragmentation**: when one request finishes and releases its memory, the next request may not fit because of size mismatch, resulting in enough total space but no way to allocate it.

This is exactly the same problem as memory management in operating systems. The OS solution is **paging** — physical memory is sliced into fixed-size pages, and logical addresses are mapped to physical pages via a page table, with no need to be contiguous. vLLM ported this idea to KV cache, calling it **PagedAttention**.

In [ ]:
# Visualizing the fragmentation problem: contiguous allocation vs paged allocation

import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contiguous allocation (left)
ax = axes[0]
ax.set_title("Contiguous allocation: heavy fragmentation", fontsize=12)

# 4 requests with 7, 13, 4, 10 blocks respectively
sizes = [7, 13, 4, 10]
labels = ['Req 1 (100t)', 'Req 2 (200t)', 'Req 3 (50t)', 'Req 4 (150t)']
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
start = 0
for size, label, color in zip(sizes, labels, colors):
    rect = patches.Rectangle((start, 0), size, 1, facecolor=color, edgecolor='black', linewidth=0.5)
    ax.add_patch(rect)
    ax.text(start + size/2, 0.5, label, ha='center', va='center', fontsize=8, color='white')
    start += size
# Suppose Req 2 finishes, freeing 13 blocks, but the next request needs 20 — does not fit
rect = patches.Rectangle((start, 0), 13, 1, facecolor='lightgray', hatch='//', edgecolor='black', linewidth=0.5)
ax.add_patch(rect)
ax.text(start + 6.5, 0.5, 'Freed by Req 2\n(unusable: next needs 20)', ha='center', va='center', fontsize=8)
start += 13

ax.set_xlim(0, start + 1)
ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('Block index (each = 16 tokens)')
ax.set_yticks([])
ax.grid(True, axis='x', alpha=0.3)

# Paged allocation (right)
ax = axes[1]
ax.set_title("Paged allocation (PagedAttention): no fragmentation", fontsize=12)

# Requests can take any blocks, no need to be contiguous
# Suppose a new request Req 5 needs 20 blocks, and can use the 13 blocks freed by Req 2 plus other free blocks
layout = [
    ('Req 1\n(7)', '#e74c3c', 7),
    ('Req 5\n(part)', '#9b59b6', 13),   # uses the 13 blocks freed by Req 2
    ('Req 3\n(4)', '#2ecc71', 4),
    ('Req 5\n(rest)', '#9b59b6', 7),    # then takes 7 more blocks
    ('Req 4\n(10)', '#f39c12', 10),
]
start = 0
for label, color, n in layout:
    rect = patches.Rectangle((start, 0), n, 1, facecolor=color, edgecolor='black', linewidth=0.5)
    ax.add_patch(rect)
    ax.text(start + n/2, 0.5, label, ha='center', va='center', fontsize=8, color='white')
    start += n

ax.set_xlim(0, start + 1)
ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('Block index (each = 16 tokens)')
ax.set_yticks([])
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observations:")
print("  Contiguous allocation: the 13 blocks freed by Req 2 cannot serve Req 5 which needs 20 -> wait or reject")
print("  Paged allocation: Req 5 takes 13 blocks (freed by Req 2) + 7 blocks (other free) = 20 blocks, immediately usable")

## 3. PagedAttention: Porting OS Paging to KV Cache

The core mechanism of PagedAttention is the **block table**: each request maintains a table that records which physical blocks its "logical KV sequence" maps to. Physical blocks can be scattered anywhere in GPU memory, with no need to be contiguous.

Concretely:
- GPU memory is sliced into fixed-size **blocks** (vLLM default is 16 tokens / block)
- Each request has a **block table**: `block_table[req_id][i]` = the physical block ID for the i-th logical block
- During attention computation, the scattered physical blocks are gathered according to the block table, and normal attention is performed

The cost is a more complex attention kernel — it must do an extra block table lookup. But the benefit far outweighs the cost: fragmentation disappears, and memory utilization rises from ~40% to over ~95%.

Below, a simplified Python simulator shows how the block allocator works.

In [ ]:
# PagedAttention simplified simulator: block-level KV cache allocation

class SimplePagedAttention:
    \"\"\"Simplified block allocator for PagedAttention.

    Each block holds a fixed number of tokens of KV cache.
    A request can take any blocks, no need to be contiguous.
    \"\"\"
    def __init__(self, num_blocks, block_size):
        self.num_blocks = num_blocks
        self.block_size = block_size
        self.free_blocks = list(range(num_blocks))  # free block list
        self.block_table = {}  # block_table[req_id] = [block_id_1, ...]
        print(f"Init: {num_blocks} blocks x {block_size} tokens/block = {num_blocks * block_size} token capacity")

    def allocate(self, req_id, num_tokens):
        \"\"\"Allocate KV cache space for a request.\"\"\"
        num_blocks_needed = (num_tokens + self.block_size - 1) // self.block_size
        if num_blocks_needed > len(self.free_blocks):
            raise MemoryError(f"Out of memory: need {num_blocks_needed} blocks, {len(self.free_blocks)} free")

        blocks = [self.free_blocks.pop(0) for _ in range(num_blocks_needed)]
        self.block_table[req_id] = blocks
        print(f"  Request {req_id}: allocated {num_blocks_needed} blocks = {blocks} (needs {num_tokens} tokens)")
        return blocks

    def free(self, req_id):
        \"\"\"Request finished, release all its blocks.\"\"\"
        blocks = self.block_table.pop(req_id, [])
        self.free_blocks.extend(blocks)
        print(f"  Request {req_id}: released {len(blocks)} blocks = {blocks}")

    def status(self):
        used = self.num_blocks - len(self.free_blocks)
        print(f"  Current: {used}/{self.num_blocks} blocks used ({used/self.num_blocks*100:.0f}%)")


print("=== PagedAttention simulation ===")
print()
pa = SimplePagedAttention(num_blocks=32, block_size=16)

print("\nPhase 1: three requests arrive together")
pa.allocate("req-A", 100)   # needs 7 blocks
pa.allocate("req-B", 50)    # needs 4 blocks
pa.allocate("req-C", 200)   # needs 13 blocks
pa.status()

print("\nPhase 2: req-B finishes, releases")
pa.free("req-B")
pa.status()

print("\nPhase 3: new request req-D arrives, needs 30 tokens = 2 blocks")
pa.allocate("req-D", 30)
pa.status()

print()
print("Key observation: req-D reuses the blocks freed by req-B — no fragmentation issue")

## 4. Continuous Batching: Dynamic Batching

The traditional batching approach (also called static batching) is: collect a batch of requests and run them together, and only admit new requests after all requests in the batch finish generating. The problem is that if short and long requests are mixed in the same batch, once the short ones finish that compute sits idle, and the next batch can only be assembled after the longest one finishes.

**Continuous batching** switches to iteration-level scheduling: after each token is generated, the current batch is inspected — finished requests are immediately removed, and newly arrived requests are immediately added. The batch size changes dynamically, and the GPU always maintains high utilization.

This idea was systematized in the Orca paper (OSDI 2022), and is implemented in nearly every modern inference engine — vLLM, SGLang, TGI, and others.

In [ ]:
# Simplified comparison of static batching vs continuous batching

# 5 requests, with different arrival times and generation lengths
requests = [
    {"id": "A", "arrival": 0, "tokens": 5},
    {"id": "B", "arrival": 0, "tokens": 8},
    {"id": "C", "arrival": 3, "tokens": 4},
    {"id": "D", "arrival": 6, "tokens": 6},
    {"id": "E", "arrival": 8, "tokens": 3},
]

# === Static batching ===
# Group requests that arrive together into a batch; cannot admit new requests until the longest in the batch finishes
def simulate_static(requests):
    time = 0
    pending = sorted(requests, key=lambda r: r["arrival"])
    timeline = []
    while pending:
        # Collect requests that can run together right now
        batch = []
        while pending and pending[0]["arrival"] <= time:
            r = pending.pop(0)
            r["remaining"] = r["tokens"]
            batch.append(r)
        if not batch:
            timeline.append([])
            time += 1
            continue
        # Run until the longest one finishes
        max_len = max(r["remaining"] for r in batch)
        for t in range(max_len):
            timeline.append([r["id"] for r in batch if r["remaining"] > 0])
            for r in batch:
                r["remaining"] -= 1
            time += 1
    return timeline

# === Continuous batching ===
def simulate_continuous(requests):
    time = 0
    active = []
    pending = sorted(requests, key=lambda r: r["arrival"])
    timeline = []
    while pending or active:
        # Admit arrived requests
        while pending and pending[0]["arrival"] <= time:
            r = pending.pop(0)
            r["remaining"] = r["tokens"]
            active.append(r)
        # Generate one step
        if active:
            timeline.append([r["id"] for r in active])
            for r in active:
                r["remaining"] -= 1
            active = [r for r in active if r["remaining"] > 0]
        else:
            timeline.append([])
        time += 1
    return timeline

# Rebuild request data (avoid mutation)
import copy
reqs1 = copy.deepcopy(requests)
reqs2 = copy.deepcopy(requests)

static_timeline = simulate_static(reqs1)
cont_timeline = simulate_continuous(reqs2)

print(f"Static batching total time: {len(static_timeline)} steps")
print(f"Continuous batching total time: {len(cont_timeline)} steps")
print(f"Speedup: {len(static_timeline) / len(cont_timeline):.2f}x")
print()

# Visualization
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 4))

color_map = {"A": "#e74c3c", "B": "#3498db", "C": "#2ecc71", "D": "#f39c12", "E": "#9b59b6"}

# Top half: static
for t, batch in enumerate(static_timeline):
    for i, rid in enumerate(batch):
        ax.barh(2, 1, left=t, color=color_map[rid], edgecolor='black', linewidth=0.3)
        if t == 0 or (t > 0 and (rid not in static_timeline[t-1] or len(static_timeline[t-1]) != len(batch))):
            ax.text(t + 0.5, 2, rid, ha='center', va='center', fontsize=8, color='white')

# Bottom half: continuous
for t, batch in enumerate(cont_timeline):
    for rid in batch:
        ax.barh(0, 1, left=t, color=color_map[rid], edgecolor='black', linewidth=0.3)
        ax.text(t + 0.5, 0, rid, ha='center', va='center', fontsize=8, color='white')

ax.set_yticks([0, 2])
ax.set_yticklabels(['Continuous', 'Static'])
ax.set_xlabel('Time step (1 step = 1 token)')
ax.set_title(f'Static ({len(static_timeline)} steps) vs Continuous ({len(cont_timeline)} steps) batching')
ax.set_xlim(-0.5, max(len(static_timeline), len(cont_timeline)) + 0.5)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("Key observation: continuous batching lets short requests finish first and new requests join immediately, shortening total time significantly")

## 5. Prefix Caching: Reusing KV Cache for Shared Prefixes

Many LLM application requests share part of their prefix. For example:
- Customer service bots: every request starts with the same system prompt
- Code assistants: every request carries the same code context
- Few-shot learning: every request carries the same examples

Computing prefill from scratch for these shared prefixes every time is wasteful — the same tokens, the same model weights, produce exactly the same KV cache. **Prefix caching** keeps this part of the KV cache so that subsequent requests with the same prefix can reuse it directly.

SGLang's **RadixAttention** uses a radix tree to organize the prefixes of all requests, automatically finding the maximum common prefix and reusing the corresponding KV cache. Anthropic's Prompt Caching is the API-layer equivalent: developers mark prefixes with `cache_control`, and on a hit the input cost drops by 90%.

In [ ]:
# Prefix caching simplified simulation: a trie organizes token sequences and finds common prefixes automatically

class SimplePrefixCache:
    \"\"\"Simplified prefix cache organized as a trie.

    Insert every request's prompt sequence into the trie; requests with the same prefix share the cache.
    \"\"\"
    def __init__(self):
        self.tree = {}  # nested dict

    def lookup_and_insert(self, tokens):
        \"\"\"Look up the cache hit length, and insert the new part into the trie.\"\"\"
        node = self.tree
        hit_len = 0
        for i, tok in enumerate(tokens):
            if tok in node:
                node = node[tok]
                hit_len += 1
            else:
                # Create new entries for the remaining tokens
                for t in tokens[i:]:
                    node[t] = {}
                    node = node[t]
                break
        return hit_len


cache = SimplePrefixCache()

# Scenario: a customer service bot, 3 users ask different questions but share the same system prompt
system_prompt_tokens = list(range(500))  # simulate a 500-token system prompt

# Request 1: system prompt + user question 1 (100 tokens)
req1 = system_prompt_tokens + [10000 + i for i in range(100)]
hit1 = cache.lookup_and_insert(req1)
print(f"Request 1 ({len(req1)} tokens): hit {hit1} tokens (full miss, all prefill)")
print(f"  Saved {hit1}/{len(req1)} = {hit1/len(req1)*100:.0f}% of prefill compute")

# Request 2: same system prompt + different question (80 tokens)
req2 = system_prompt_tokens + [20000 + i for i in range(80)]
hit2 = cache.lookup_and_insert(req2)
print(f"\nRequest 2 ({len(req2)} tokens): hit {hit2} tokens (system prompt fully hit)")
print(f"  Saved {hit2}/{len(req2)} = {hit2/len(req2)*100:.0f}% of prefill compute")

# Request 3: same system prompt + yet another question (120 tokens)
req3 = system_prompt_tokens + [30000 + i for i in range(120)]
hit3 = cache.lookup_and_insert(req3)
print(f"\nRequest 3 ({len(req3)} tokens): hit {hit3} tokens")
print(f"  Saved {hit3}/{len(req3)} = {hit3/len(req3)*100:.0f}% of prefill compute")

print()
print("Key observation: requests with the same system prompt only need to prefill the user question part after the prefix hits")
print(f"Across 3 requests, total prefill is {(len(req1)-hit1) + (len(req2)-hit2) + (len(req3)-hit3)} tokens")
print(f"Without caching, it would be {len(req1) + len(req2) + len(req3)} tokens of prefill")

## 6. Prefill / Decode Disaggregation

LLM inference has two phases with **completely opposite resource usage patterns**:

| Phase | Compute characteristic | Bottleneck | Typical operation |
|:---|:---|:---|:---|
| **Prefill** | Process the entire prompt in one shot | Compute-bound (compute) | Process a 1K-32K token input |
| **Decode** | Generate one new token at a time | Memory-bound (memory bandwidth) | Generate output tokens one by one |

The prefill phase saturates compute and underutilizes bandwidth; the decode phase saturates bandwidth and underutilizes compute. If both kinds of requests run in the same cluster, resource conflicts arise — prefill requests and decode requests each wait for resources they do not lack, and GPU utilization stays low.

**Prefill/Decode disaggregation** deploys the two phases in different clusters. The prefill cluster uses high-compute machines (such as H100 SXM5), and the decode cluster uses high-bandwidth machines. After prefill completes, the KV cache is transferred over the network to the decode cluster to continue generation. Mooncake (the inference framework behind Moonshot/Kimi) and DistServe both adopt this architecture, with papers reporting throughput improvements of 50%-150%.

The engineering difficulty is KV cache transfer — a 32K-token KV cache can be several GB, and cross-machine transfer latency cannot be ignored. Mooncake solves this with a global KV cache pool plus RDMA networking.

In [ ]:
# Resource usage patterns of prefill vs decode

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Resource usage comparison
ax = axes[0]
modes = ['Prefill\n(compute-bound)', 'Decode\n(memory-bound)']
compute_use = [90, 25]
memory_use = [40, 95]
x = range(len(modes))
width = 0.35
ax.bar([i - width/2 for i in x], compute_use, width, label='Compute utilization', color='#3498db')
ax.bar([i + width/2 for i in x], memory_use, width, label='Memory BW utilization', color='#e74c3c')
ax.set_xticks(list(x))
ax.set_xticklabels(modes)
ax.set_ylabel('Utilization (%)')
ax.set_title('Prefill and Decode have opposite resource usage patterns')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(0, 110)

# Mixed vs disaggregated
ax = axes[1]
configs = ['Mixed\n(same cluster)', 'Disaggregated\n(separated)']
throughputs = [100, 240]
bars = ax.bar(configs, throughputs, color=['#95a5a6', '#2ecc71'])
ax.set_ylabel('Relative throughput (tokens/s)')
ax.set_title('Disaggregated serving throughput improvement\n(DistServe/Mooncake reported values)')
ax.grid(True, axis='y', alpha=0.3)
for i, v in enumerate(throughputs):
    ax.text(i, v + 5, f'{v}', ha='center', fontsize=11)
ax.set_ylim(0, 280)

plt.tight_layout()
plt.show()

print("Key observation: prefill/decode disaggregation lets each compute mode reach its maximum utilization")
print("Cost: KV cache must be transferred between the prefill and decode clusters, raising engineering complexity")

## 7. Comparison of Mainstream Inference Engines

The techniques above are packaged into production-grade inference engines. The mainstream ones today:

| Engine | Origin | Core feature | Typical scenario |
|:---|:---|:---|:---|
| **vLLM** | UC Berkeley | PagedAttention + continuous batching | General purpose, most active community |
| **SGLang** | UC Berkeley | RadixAttention (prefix-tree cache) + structured generation | Multi-turn dialogue, agentic, complex prefix reuse |
| **TensorRT-LLM** | NVIDIA | Best-in-class FP8 kernels on Hopper/Blackwell | Chasing peak performance on NVIDIA cards |
| **LMDeploy** | OpenMMLab | Turbomind + W4A16 | Friendly to the Chinese model ecosystem |
| **TGI** | HuggingFace | Tight integration with HF Hub | HF ecosystem users |
| **Mooncake** | Moonshot/Kimi | Centralized KV cache + prefill/decode disaggregation | Long-context high-concurrency |

Selection guidance: for general scenarios, vLLM is the safe default; for agentic / multi-turn scenarios, SGLang is more suitable; for peak performance on NVIDIA cards, TensorRT-LLM; for Chinese models (Qwen, GLM, DeepSeek), LMDeploy has good integration; for DeepSeek-family models (which use MLA), both vLLM and SGLang support them, but engineering maturity is still catching up.

Actual deployment also depends on model architecture. Models with non-standard components such as MLA, Mamba, or linear attention need confirmation that the inference engine supports them natively, otherwise a fallback path is required and performance drops.

## Summary

- [ ] Core metrics of LLM serving: throughput, TTFT, TPOT, and concurrency, which constrain each other
- [ ] The bottleneck of LLM inference is memory bandwidth rather than compute — this is why KV cache size matters so much
- [ ] Contiguous allocation under multiple requests causes internal and external fragmentation; PagedAttention solves it with a block table
- [ ] Continuous batching dynamically forms batches at the iteration level, letting short requests finish first and new requests join immediately
- [ ] Prefix caching (RadixAttention) uses a trie to organize request sequences and automatically reuses the KV cache of shared prefixes
- [ ] Prefill is compute-bound and decode is memory-bound; disaggregated deployment (Mooncake/DistServe) significantly improves throughput
- [ ] Mainstream engines: vLLM (general), SGLang (agentic), TensorRT-LLM (peak NVIDIA), LMDeploy (Chinese models)

References: [vLLM/PagedAttention](https://arxiv.org/abs/2309.06180), [SGLang/RadixAttention](https://arxiv.org/abs/2312.07104), [Orca/Continuous batching](https://www.usenix.org/con/osdi22/presentation/yu), [DistServe](https://arxiv.org/abs/2401.09670), [Mooncake](https://arxiv.org/abs/2407.00079).

## Exercises

> You can ask AI to help explain the approach, but it is not recommended to have AI "solve the exercise for you" directly.

**Exercise 1: KV Cache Capacity Calculation**

Given: A100 80GB, Llama-7B FP16 (weights 13 GB), block_size = 16 tokens. The remaining memory is all used for KV cache. Calculate: at 32K context, what is the maximum number of concurrent requests that can be served simultaneously? (Only consider the KV cache limit, ignore other overhead.)

Hint: Refer to the calculation in Section 1. Each concurrent request needs `seq_len x num_layers x num_heads x head_dim x 2 x bytes_per_element` bytes of KV cache. Remaining memory = 80 - 13 = 67 GB.

In [ ]:
# Exercise 1: KV Cache capacity calculation

gpu_total_gb = 80
model_weights_gb = 13
block_size = 16
seq_len = 32768
num_layers = 32
num_heads = 32
head_dim = 128
bytes_per_element = 2  # FP16

# TODO: fill in the calculation
available_gb = None            # gpu_total_gb - model_weights_gb
kv_cache_per_request_gb = None # seq_len x num_layers x num_heads x head_dim x 2 x 2 / 1024**3
max_concurrency = None         # available_gb / kv_cache_per_request_gb (floor)

assert available_gb is not None, "Please compute available_gb first"
assert kv_cache_per_request_gb is not None, "Please compute kv_cache_per_request_gb first"
assert max_concurrency is not None, "Please compute max_concurrency first"

expected_avail = 80 - 13
expected_kv = 32768 * 32 * 32 * 128 * 2 * 2 / (1024**3)
expected_max = int(expected_avail / expected_kv)

assert abs(available_gb - expected_avail) < 0.01
assert abs(kv_cache_per_request_gb - expected_kv) < 0.01
assert max_concurrency == expected_max

print(f"✅ Exercise 1 passed")
print(f"   Available memory: {available_gb} GB")
print(f"   KV cache per request: {kv_cache_per_request_gb:.2f} GB")
print(f"   Max concurrency: {max_concurrency} requests")
print(f"   Key observation: this is why PagedAttention + continuous batching are needed — to push concurrency to the limit")

**Exercise 2: Implement the allocate Method of Simplified PagedAttention**

Below is a simplified `SimplePagedAttention` class with the implementation of the `allocate` method omitted. Complete it: given the number of tokens a request needs, compute the number of blocks, take them from `free_blocks`, and record them in `block_table`.

Hint: number of blocks needed = `(num_tokens + block_size - 1) // block_size` (round up). If `free_blocks` is insufficient, raise MemoryError.

In [ ]:
# Exercise 2: Complete PagedAttention's allocate

class SimplePagedAttention:
    def __init__(self, num_blocks, block_size):
        self.num_blocks = num_blocks
        self.block_size = block_size
        self.free_blocks = list(range(num_blocks))
        self.block_table = {}

    def allocate(self, req_id, num_tokens):
        # TODO: complete this method
        # 1. Compute how many blocks are needed (round up)
        # 2. If not enough, raise MemoryError
        # 3. Take the corresponding number of blocks from free_blocks
        # 4. Record them in block_table[req_id]
        # 5. return these blocks
        pass


# Verification
pa = SimplePagedAttention(num_blocks=10, block_size=16)
b1 = pa.allocate("A", 50)
b2 = pa.allocate("B", 20)
assert len(b1) == 4, f"A needs 4 blocks (50/16 rounded up), got {len(b1)}"
assert len(b2) == 2, f"B needs 2 blocks (20/16 rounded up), got {len(b2)}"
assert len(pa.free_blocks) == 4, f"Should have 4 blocks left, got {len(pa.free_blocks)}"

# The third request needs more blocks than remain
try:
    pa.allocate("C", 200)  # needs 13 blocks but only 4 left
    assert False, "Should raise MemoryError"
except MemoryError:
    pass

print(f"✅ Exercise 2 passed")
print(f"   A allocated: {b1}")
print(f"   B allocated: {b2}")
print(f"   Remaining free blocks: {pa.free_blocks}")

**Exercise 3: Prefix Cache Hit Rate Calculation**

Given the token sequences of a set of requests (each is a list of token ids), use the provided `SimplePrefixCache` to compute the **overall hit rate** after all requests are processed (hit tokens / total tokens).

Hint: call `lookup_and_insert` in turn and accumulate the returned hit length each time.

In [ ]:
# Exercise 3: Prefix cache hit rate calculation

class SimplePrefixCache:
    def __init__(self):
        self.tree = {}

    def lookup_and_insert(self, tokens):
        node = self.tree
        hit_len = 0
        for i, tok in enumerate(tokens):
            if tok in node:
                node = node[tok]
                hit_len += 1
            else:
                for t in tokens[i:]:
                    node[t] = {}
                    node = node[t]
                break
        return hit_len


# 3 requests; the first is a base prompt, the latter two share a prefix
system_prompt = list(range(200))   # 200-token system prompt
req1 = system_prompt + [1000 + i for i in range(50)]
req2 = system_prompt + [2000 + i for i in range(80)]
req3 = system_prompt + [3000 + i for i in range(100)]

all_reqs = [req1, req2, req3]

cache = SimplePrefixCache()

# TODO: compute total hits and total tokens
total_hits = None     # sum(cache.lookup_and_insert(r) for r in all_reqs)
total_tokens = None   # sum(len(r) for r in all_reqs)
hit_rate = None       # total_hits / total_tokens

assert total_hits is not None
assert total_tokens is not None
assert hit_rate is not None

# Verification
expected_total = sum(len(r) for r in all_reqs)
# req1 is a full miss (200 + 50 = 250 tokens, all newly inserted, hit=0 because first insert)
# Wait — lookup_and_insert also returns hit_len on the first insert, i.e. the matched part
# req1: brand-new insert, hit=0
# req2: first 200 tokens already exist, hit=200
# req3: first 200 tokens already exist, hit=200
expected_hits = 0 + 200 + 200
expected_rate = expected_hits / expected_total

assert total_tokens == expected_total, f"Should be {expected_total}"
assert total_hits == expected_hits, f"Should be {expected_hits}"
assert abs(hit_rate - expected_rate) < 0.001

print(f"✅ Exercise 3 passed")
print(f"   Total tokens: {total_tokens}")
print(f"   Hit tokens: {total_hits}")
print(f"   Hit rate: {hit_rate*100:.1f}%")
print(f"   Key observation: the more requests share a prefix, the higher the hit rate, and the more prefill compute is saved")